# Урок 28. Большие языковые модели: как они работают и как проверять их ответы

**Дата:** 01.04.2027 · **60 минут** · 8–9 класс  
**Модуль 5.** ИИ-приложения (уроки 26–29)  
**Опирается на:** урок 2 (промпты), урок 8 (словари), урок 27 (текст как данные), уроки 19–21 (обучение на примерах, ошибки моделей).

ChatGPT, GigaChat, YandexGPT, Claude — **большие языковые модели** (*LLM, large language models*). Они пишут тексты, объясняют задачи, пишут код. На уроке 2 мы учились ставить им задачи. Сегодня заглянем внутрь: соберём **игрушечную языковую модель** своими руками, поймём, почему LLM уверенно ошибаются, и научимся **проверять** их ответы.

## Цели урока

К концу занятия ты сможешь:

1. Объяснять, как языковая модель генерирует текст: **предсказание следующего слова**.
2. Собрать игрушечную языковую модель на словаре.
3. Объяснять, что такое **галлюцинация** и почему она возникает.
4. Проверять ответы LLM: факты, числа, код, источники.
5. Пользоваться чек-листом проверки перед тем, как доверять ответу.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–15 | Как устроена языковая модель; игрушечная модель |
| 15–25 | Токены, обучение, ограничения LLM |
| 25–35 | Галлюцинации: почему модель уверенно ошибается |
| 35–55 | Практика: проверяем «ответы модели» |
| 55–60 | Итог и домашнее задание |

## 1. Главная идея: угадать следующее слово

Языковая модель решает одну задачу: **по началу текста предсказать, какое слово (точнее, кусочек слова) будет следующим**.

> «Мама мыла …» → скорее всего «раму».  
> «Столица Франции — …» → скорее всего «Париж».

Чтобы сгенерировать ответ, модель повторяет это много раз: предсказала слово → добавила его к тексту → предсказывает следующее → и так далее.

Это та же **классификация** из нашего курса — только классов не 2 и не 10, а десятки тысяч: все слова и кусочки слов, которые знает модель.

## 2. Игрушечная языковая модель

Соберём самую простую языковую модель. Она будет помнить **только предыдущее слово** и знать, какие слова шли после него в обучающем тексте.

**Обучение** — пройти по тексту и для каждого слова записать в словарь, какое слово шло следом. Если после «данные» дважды шло «нужно», в списке «нужно» будет два раза — так модель «запомнит», что это частое продолжение.

In [ ]:
import random

corpus = (
    "модель учится на данных . модель ошибается на новых данных . "
    "данные нужно чистить . данные нужно проверять . "
    "инженер проверяет модель . инженер чистит данные . "
    "модель учится на примерах . инженер учится на ошибках ."
)

words = corpus.split()
next_words = {}
for i in range(len(words) - 1):
    current = words[i]
    following = words[i + 1]
    if current not in next_words:
        next_words[current] = []
    next_words[current].append(following)

print("После «модель»:", next_words["модель"])
print("После «данные»:", next_words["данные"])
print("После «инженер»:", next_words["инженер"])

**Генерация** — начать с какого-нибудь слова и много раз выбирать **случайное** продолжение из списка. Частые продолжения встречаются в списке чаще — значит, и выбираются чаще.

In [ ]:
def generate(start, length, seed):
    random.seed(seed)
    text = [start]
    word = start
    for step in range(length):
        if word not in next_words:
            break
        word = random.choice(next_words[word])
        text.append(word)
    return " ".join(text)

for seed in [2, 6, 7, 11]:
    print(generate("инженер", 10, seed))

Посмотри на результаты:

- часть фраз звучит разумно;
- часть — бессмыслица из склеенных кусков: «инженер проверяет модель учится на новых данных», «инженер чистит данные нужно чистить»;
- а фразы «модель учится на ошибках» **не было** в обучающем тексте! Были «инженер учится на ошибках» и «модель учится на примерах» — модель склеила из них **новое утверждение**, которое звучит гладко и уверенно.

Модель не знает, **правда** ли это, — она знает только, какие слова **часто стоят рядом**. Именно так в миниатюре выглядит галлюцинация.

Это главное, что нужно понять про любые языковые модели: они учатся **правдоподобию**, а не **истине**.

## 3. Чем настоящие LLM отличаются от игрушки

| | Наша игрушка | Настоящая LLM |
|--|--------------|---------------|
| Обучающий текст | 4 строки | огромная часть интернета, книги, код — триллионы слов |
| Учитывает | 1 предыдущее слово | тысячи предыдущих слов (весь диалог) |
| Как устроена | словарь со списками | нейросеть с миллиардами параметров |
| После обучения | — | дообучение на примерах «вопрос → хороший ответ», чтобы быть полезным помощником |

Но принцип тот же: **продолжить текст правдоподобно**.

### Токены

LLM работают не со словами, а с **токенами** — кусочками слов. «Инженерия» может разбиться на «Инж», «енер», «ия». Поэтому модели иногда плохо считают буквы в словах и ошибаются в арифметике: они видят не буквы и цифры по отдельности, а кусочки.

### Ограничения, о которых надо помнить

- **Дата среза знаний.** Модель знает только то, что было в обучающих текстах до определённого момента. О свежих событиях она может не знать — или выдумать.
- **Нет «памяти» о тебе** между разговорами (если сервис специально не сохраняет историю).
- **Нет доступа к интернету**, если его не дали как инструмент (тогда это уже агент, урок 2).
- **Длина контекста.** Очень длинный разговор модель «забывает» с начала.

## 4. Галлюцинации

**Галлюцинация** — уверенный, правдоподобный, но **неверный** ответ языковой модели.

Почему они возникают:

1. Модель обучена продолжать текст правдоподобно. Ответ «не знаю» в обучающих текстах встречается редко — правдоподобнее выдать что-то похожее на ответ.
2. Редкие факты (малоизвестный человек, маленький город, точная дата) модель видела мало раз и «смешивает» с похожими.
3. Числа, цитаты, ссылки на книги и сайты выглядят как узор из символов — модель может сгенерировать узор, который похож на настоящий, но не существует.

**Где галлюцинации особенно часты:**

| Что | Как проявляется |
|-----|-----------------|
| Ссылки и источники | несуществующие статьи, книги, адреса сайтов |
| Цитаты | «Как сказал Эйнштейн…» — а он этого не говорил |
| Числа и даты | неточные годы, статистика «с потолка» |
| Код | функции и параметры, которых нет в библиотеке; код без проверки крайних случаев |
| Узкие факты | биографии малоизвестных людей, местные события |

## 5. Чек-лист проверки ответа LLM

1. **Факты** — проверь по **двум** независимым надёжным источникам (учебник, энциклопедия, официальный сайт).
2. **Числа** — пересчитай сам (вручную или кодом).
3. **Код** — **запусти**, проверь на крайних случаях (пустой список, ноль, отрицательные числа).
4. **Ссылки** — открой и убедись, что страница существует и там написано то же самое.
5. **Цитаты** — найди первоисточник.
6. **Понимание** — можешь ли ты объяснить ответ своими словами? Если нет — он ещё не твой (урок 2).

Полезно спрашивать модель: «Насколько ты уверен?», «На чём основан ответ?», «Покажи вычисления по шагам». Но и ответ на эти вопросы может быть выдумкой — **проверка всё равно на тебе**.

## 6. Практика: проверяем «ответы модели»

Ниже — ответы, которые могла бы выдать языковая модель. Некоторые верны, некоторые — нет. Твоя задача — **проверить**, а не поверить.

### Практика 1. Код от модели (8 минут)

Ученик попросил: «Напиши функцию, которая считает средний балл». Модель ответила:

In [ ]:
def average(scores):
    total = 0
    for s in scores:
        total += s
    return total / len(scores)

print(average([4, 5, 3]))
print(average([5, 5, 4, 4]))

На первый взгляд работает. Проверь **крайний случай** — пустой список. Запусти ячейку ниже.

In [ ]:
# Здесь намеренная ошибка: проверяем крайний случай
print(average([]))

Код выглядел правильным, но падает на пустом списке. Модель не соврала специально — она выдала **типичный** код, а крайние случаи в типичном коде часто забывают.

**Задание:** исправь функцию так, чтобы для пустого списка она возвращала `None` (как на уроке 7), и проверь её на трёх случаях.

In [ ]:
# Практика 1: исправленная функция и тесты

<details>
<summary>Решение</summary>

```python
def average(scores):
    if len(scores) == 0:
        return None
    total = 0
    for s in scores:
        total += s
    return total / len(scores)

print(average([4, 5, 3]))
print(average([5, 5, 4, 4]))
print(average([]))
```
</details>

### Практика 2. Число от модели (5 минут)

Вопрос: «Сколько секунд в неделе?» Ответ модели: «В неделе 640 800 секунд». Проверь кодом.

In [ ]:
# Практика 2

<details>
<summary>Решение</summary>

```python
print(7 * 24 * 60 * 60)
```
Правильно: 604 800. Ответ модели правдоподобен (шестизначное число, начинается на 6), но неверен.
</details>

### Практика 3. Верно, сомнительно или неверно? (10 минут)

Оцени утверждения «модели» по знаниям из нашего курса. Поставь **В** (верно), **Н** (неверно) или **?** (надо проверять по источникам) и объясни.

| № | Утверждение «модели» | В / Н / ? | Почему |
|--:|----------------------|:---------:|--------|
| 1 | Accuracy — всегда лучшая метрика для любой задачи классификации. | | |
| 2 | Тестовая выборка нужна, чтобы честно оценить модель на новых данных. | | |
| 3 | Линейная регрессия всегда лучше дерева решений. | | |
| 4 | В sklearn у модели kNN есть параметр `n_neighbors`. | | |
| 5 | Если модель показывает 100% на train, она точно будет хороша на новых данных. | | |
| 6 | Python создал Гвидо ван Россум, первая версия вышла в 1991 году. | | |
| 7 | Согласно исследованию МГУ 2019 года, 87% школьников учатся лучше с ИИ. | | |

<details>
<summary>Ответы</summary>

1. **Н** — при несбалансированных классах accuracy обманывает (урок 21).
2. **В** — урок 19.
3. **Н** — всё зависит от данных; сравнивают на validation и test (урок 23).
4. **В** — можно проверить запуском или в документации (урок 20).
5. **Н** — это может быть переобучение (урок 23).
6. **В**, но для итоговой работы такое проверяют по источнику (урок 3 + энциклопедия).
7. **?** — типичный признак галлюцинации: конкретное число, громкое учреждение, нет ссылки. Пока не найден первоисточник — не верить и не цитировать.
</details>

### Практика 4. Проверка с настоящим помощником (10 минут, если есть доступ)

1. Задай ИИ-помощнику вопрос, ответ на который **ты знаешь точно** (например, по материалам нашего курса). Проверь ответ.
2. Задай вопрос-ловушку: про несуществующее событие или книгу (например, «Расскажи о книге „Python для котов“ Ивана Петрова 2015 года»). Выдумал ли помощник подробности или честно сказал, что не знает?
3. Попроси объяснить решение задачи с числами и **пересчитай** сам.

| Вопрос | Ответ помощника (кратко) | Моя проверка | Итог: верно / неверно / выдумка / честное «не знаю» |
|--------|--------------------------|--------------|------------------------------------------------------|
| | | | |
| | | | |
| | | | |

## 7. Как инженеру жить с LLM

LLM — мощный инструмент, если помнить его природу:

- **Используй** для: черновиков, объяснений, идей, поиска ошибок в коде, генерации тестовых примеров.
- **Проверяй** всё, что идёт в отчёт, проект или решение: факты, числа, код, ссылки.
- **Не отправляй** в чат личные данные и чужие работы (урок 24).
- **Указывай**, где и как помогал ИИ: это честно и в учёбе, и в работе.
- **Отвечаешь за результат ты**, а не модель.

## Мини-глоссарий урока 28

| Термин | Смысл |
|--------|-------|
| **LLM** | большая языковая модель: предсказывает следующий кусочек текста |
| **Токен** | кусочек слова, с которым работает модель |
| **Генерация** | повторное предсказание следующего токена |
| **Галлюцинация** | уверенный, но неверный ответ |
| **Дата среза знаний** | после неё модель ничего не знает о мире |
| **Верификация** | проверка ответа по независимым источникам, расчётом или запуском |

## Итог урока

1. Языковая модель продолжает текст **правдоподобно** — правдоподобие не гарантирует истину.
2. Галлюцинации чаще всего в фактах, числах, ссылках, цитатах и коде.
3. Ответ LLM становится знанием только после **проверки**: источники, расчёт, запуск, понимание.

## Домашнее задание (20 минут)

1. Дополни `corpus` игрушечной модели своими 5–6 предложениями на одну тему и сгенерируй 5 текстов. Найди среди них пример «правдоподобной бессмыслицы».
2. Запиши 3 своих правила проверки ответов LLM — своими словами, на примерах из жизни.

In [ ]:
# Домашнее задание

## Самопроверка

1. Какую задачу решает языковая модель при генерации текста?
2. Почему игрушечная модель выдаёт грамматичную бессмыслицу?
3. Что такое галлюцинация? Где она встречается чаще всего?
4. Как проверить код, который написала модель?
5. Почему ответ «не знаю» у LLM встречается реже, чем хотелось бы?

**Следующий урок (08.04.2027):** ИИ как помощник инженера — как работать с ИИ на итоговом проекте.